# Notebook 1 (MASTER) — Data Prep for ALL datasets
Runs the *exact* leakage-audit / split / scale logic of the original NB1, but looped over every dataset in the config table. Local / Anaconda (no Colab).

**Edit the CONFIG cell**, then Run All. Each dataset's outputs are saved separately.

In [1]:
import os, json, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')  # headless-safe; remove for inline plots
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer

## CONFIG — edit BENCH_ROOT and the per-dataset table

In [2]:
import sklearn

# Global configuration to keep DataFrame structure and column names
sklearn.set_config(transform_output="pandas")
BENCH_ROOT = r'C:\Users\tanvi\Downloads\HealthSec_FS'   # <-- your local root
RANDOM_STATE = 42
CHUNK_ROWS   = 200_000
LEAK_PURITY  = 0.999   # a column this predictive of the label is treated as leakage

# Per-dataset settings. VERIFY label_col / benign_keyword against each raw CSV header.
DATASETS = {
  'CICIOMT': dict(folder='CICIOMT', data_file='', label_col='label', benign_keyword='benign',
                  sampling='balanced_floor', benign_target=2000, attack_per_benign=1.0, target_subsample=20000),
  'WUSTL':   dict(folder='WUSTL', data_file='', label_col='Label', benign_keyword='0',
                  sampling='proportional', benign_target=14000, attack_per_benign=1.0, target_subsample=16000),

  'HIPAA':   dict(folder='HIPAA', data_file='', label_col='is_attack', benign_keyword='0',
                  sampling='none', benign_target=2310, attack_per_benign=1.0, target_subsample=3300),
  'WUSTL_HDRL': dict(folder='WUSTL_HDRL', data_file='', label_col='Label', benign_keyword='0',
                  sampling='proportional', benign_target=27000, attack_per_benign=1.0, target_subsample=30000),
}
ACTIVE = ['CICIOMT','WUSTL','HIPAA','WUSTL_HDRL']   # datasets to process this run
TARGET_COLUMN = 'is_attack'
SUBFOLDERS = ['data','processed','features','eval','results']

def paths_for(key):
    root=os.path.join(BENCH_ROOT, DATASETS[key]['folder']); P={'root':root}
    for s in SUBFOLDERS:
        p=os.path.join(root,s); os.makedirs(p,exist_ok=True); P[s]=p
    return P

## Per-dataset pipeline (original NB1 logic, unchanged)

In [3]:
def run_dataset(KEY):
    d = DATASETS[KEY]; PATHS = paths_for(KEY)
    DATA_FILE      = d['data_file']
    LABEL_COL      = d['label_col']
    BENIGN_KEYWORD = d['benign_keyword']
    SAMPLING       = d['sampling']
    BENIGN_TARGET  = d['benign_target']
    ATTACK_PER_BENIGN = d['attack_per_benign']
    TARGET_SUBSAMPLE  = d['target_subsample']
    print("="*64); print(f"NB1 :: {KEY}  (root={PATHS['root']})"); print("="*64)

    # ---- locate CSV ----
    def _find_csv():
        if DATA_FILE: return os.path.join(PATHS['data'], DATA_FILE)
        cand=[f for f in os.listdir(PATHS['data']) if f.lower().endswith('.csv')]
        if not cand: raise FileNotFoundError(f"No CSV in {PATHS['data']}")
        cand.sort(key=lambda f: os.path.getmtime(os.path.join(PATHS['data'],f)), reverse=True)
        return os.path.join(PATHS['data'], cand[0])
    RAW_CSV=_find_csv(); print("Using:", os.path.basename(RAW_CSV))

    def _is_attack(s):
        return (~s.astype(str).str.contains(str(BENIGN_KEYWORD), case=False, na=False)).astype(int)

    # ---- PASS 1: class counts over chunks ----
    label_counts, n0, n1 = {}, 0, 0
    for chunk in pd.read_csv(RAW_CSV, usecols=[LABEL_COL], chunksize=CHUNK_ROWS):
        vc=chunk[LABEL_COL].value_counts()
        for k,v in vc.items(): label_counts[k]=label_counts.get(k,0)+int(v)
        ia=_is_attack(chunk[LABEL_COL]); n0+=int((ia==0).sum()); n1+=int((ia==1).sum())
    TOTAL=n0+n1
    print(f"Total rows: {TOTAL:,} | benign={n0:,} attack={n1:,}")
    if n0==0 or n1==0:
        raise ValueError(f"benign_keyword '{BENIGN_KEYWORD}' split failed (benign={n0}, attack={n1}). Fix config for {KEY}.")

    # ---- PASS 2: subsample ----
    if SAMPLING=='balanced_floor':
        keep_b=min(n0, BENIGN_TARGET); keep_a=min(n1, int(keep_b*ATTACK_PER_BENIGN))
        fb = keep_b/n0 if n0 else 0; fa = keep_a/n1 if n1 else 0
    else:
        fr=min(1.0,(TARGET_SUBSAMPLE or TOTAL)/TOTAL); fb=fa=fr
    rng=np.random.RandomState(RANDOM_STATE); parts=[]
    for chunk in pd.read_csv(RAW_CSV, chunksize=CHUNK_ROWS):
        chunk[TARGET_COLUMN]=_is_attack(chunk[LABEL_COL])
        for cls,frac in ((0,fb),(1,fa)):
            sub=chunk[chunk[TARGET_COLUMN]==cls]
            if frac<1.0 and len(sub):
                sub=sub.sample(frac=frac, random_state=RANDOM_STATE)
            parts.append(sub)
    df=pd.concat(parts, ignore_index=True)
    print(f"Subsample: {len(df):,} rows")

    # ---- leakage / identifier audit ----
    audit={}
    def _flag(cols,reason):
        for c in cols:
            if c in df.columns and c!=TARGET_COLUMN: audit[c]=reason
    _flag([LABEL_COL],'raw label (binarised)')
    y=df[TARGET_COLUMN]
    for c in df.columns:
        if c in (TARGET_COLUMN,LABEL_COL) or c in audit: continue
        nu=df[c].nunique(dropna=False)
        if nu<=1: audit[c]='constant'; continue
        if nu>0.9*len(df): audit[c]='id-like (near-unique)'; continue
        # purity: does any single value near-perfectly predict the label?
        try:
            purity=df.groupby(c)[TARGET_COLUMN].agg(lambda s: s.value_counts(normalize=True).max()).mean()
            if purity>LEAK_PURITY: audit[c]=f'label proxy (purity={purity:.3f})'
        except Exception: pass
    cols=[c for c in audit if c in df.columns]; df=df.drop(columns=cols)
    print(f"Dropped {len(cols)} audited columns -> shape {df.shape}")

    # ---- coerce numeric ----
    for c in df.columns:
        if c!=TARGET_COLUMN: df[c]=pd.to_numeric(df[c], errors='coerce')

    # ---- drop columns that are completely NaN after coercion ----
    df = df.dropna(how='all', axis=1)

    # ---- split ----
    feat=[c for c in df.columns if c!=TARGET_COLUMN]
    X=df[feat].copy(); y=df[TARGET_COLUMN].copy()
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.20,random_state=RANDOM_STATE,stratify=y)

    # ---- impute (fit on TRAIN only) ----
    imp=SimpleImputer(strategy='median').fit(Xtr)
    feat=list(imp.get_feature_names_out(feat))  # update column list to match imputer output
    Xtr=pd.DataFrame(imp.transform(Xtr),columns=feat,index=Xtr.index)
    Xte=pd.DataFrame(imp.transform(Xte),columns=feat,index=Xte.index)

    # ---- scale continuous (fit on TRAIN only; skip binary) ----
    binary_like=[c for c in Xtr.columns if set(pd.unique(Xtr[c])).issubset({0,1})]
    cont=[c for c in Xtr.columns if c not in binary_like]
    if cont:
        sc=MinMaxScaler().fit(Xtr[cont])
        Xtr[cont]=sc.transform(Xtr[cont]); Xte[cont]=sc.transform(Xte[cont])
    else:
        sc=None

    # ---- save ----
    tr=Xtr.copy(); tr[TARGET_COLUMN]=ytr.values
    te=Xte.copy(); te[TARGET_COLUMN]=yte.values
    tr.to_parquet(os.path.join(PATHS['processed'],'train.parquet'))
    te.to_parquet(os.path.join(PATHS['processed'],'test.parquet'))
    if sc is not None: joblib.dump(sc, os.path.join(PATHS['processed'],'scaler.joblib'))
    joblib.dump(imp, os.path.join(PATHS['processed'],'imputer.joblib'))
    meta=dict(dataset=KEY, source_file=os.path.basename(RAW_CSV), target_column=TARGET_COLUMN,
              n_features=int(Xtr.shape[1]), subsample_rows=int(len(df)), full_rows=int(TOTAL),
              dropped_columns=audit, random_state=RANDOM_STATE)
    json.dump(meta, open(os.path.join(PATHS['processed'],'meta.json'),'w'), indent=2, default=str)
    print(f"[OK] {KEY}: {Xtr.shape[1]} features | train {len(Xtr):,} / test {len(Xte):,}")
    return meta

## Run all active datasets

In [4]:
for _ds in ACTIVE:
    try:
        run_dataset(_ds)
    except Exception as e:
        print(f'[ERROR] NB1 {_ds}: {e}')
print('\nNB1 master complete.')

NB1 :: CICIOMT  (root=C:\Users\tanvi\Downloads\HealthSec_FS\CICIOMT)
Using: CIC_IoMT_2024_WiFi_MQTT_test.csv
Total rows: 1,614,182 | benign=37,607 attack=1,576,575
Subsample: 4,002 rows
Dropped 12 audited columns -> shape (4002, 35)
[OK] CICIOMT: 34 features | train 3,201 / test 801
NB1 :: WUSTL  (root=C:\Users\tanvi\Downloads\HealthSec_FS\WUSTL)
Using: wustl-ehms-2020_with_attacks_categories.csv
Total rows: 16,318 | benign=14,272 attack=2,046
Subsample: 16,000 rows
Dropped 16 audited columns -> shape (16000, 30)
[OK] WUSTL: 28 features | train 12,800 / test 3,200
NB1 :: HIPAA  (root=C:\Users\tanvi\Downloads\HealthSec_FS\HIPAA)
Using: breach_report (01-01-2021 to 21-05-2026).csv
Total rows: 3,293 | benign=697 attack=2,596
Subsample: 3,293 rows
Dropped 0 audited columns -> shape (3293, 33)
[OK] HIPAA: 32 features | train 2,634 / test 659
NB1 :: WUSTL_HDRL  (root=C:\Users\tanvi\Downloads\HealthSec_FS\WUSTL_HDRL)
Using: wustl_hdrl_2024.csv
Total rows: 145,123 | benign=132,884 attack=12,23